# Query Expansion (QE) Experiment for Indonesian Code Search

This notebook runs Query Expansion experiments on COSQA with Indonesian translations using coir-eval.

## Setup

Run the cells below in order. The notebook will:
1. Install coir-eval from pip
2. Load COSQA from HuggingFace
3. Load Indonesian translations (upload `cosqa_queries_indonesian.csv` to Colab first)
4. Run Query Expansion experiments
5. Compare English vs Indonesian performance

In [ ]:
# Install dependencies
!pip install torch numpy pandas scikit-learn tqdm transformers sentence-transformers datasets faiss-cpu

In [ ]:
# Install coir-eval from pip
!pip install coir-eval

In [ ]:
# Import coir modules
import sys
import json
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Import coir modules
from coir.data_loader import load_data_from_hf
from coir.embedding_expander import CrossLingualEmbeddingExpander
from coir.dense_retriever import DenseRetriever, BM25Retriever

print("Successfully imported coir modules!")

## Load Data

In [ ]:
def load_cosqa_data():
    """Load COSQA dataset with Indonesian translations."""
    # Load Indonesian translations if file exists
    translations_file = Path("cosqa_queries_indonesian.csv")
    translations = {}
    
    if translations_file.exists():
        trans_df = pd.read_csv(translations_file, sep="|")
        translations = dict(zip(trans_df["qid"], trans_df["query_id"]))
        print(f"Loaded {len(translations)} Indonesian translations")
    else:
        print("Note: cosqa_queries_indonesian.csv not found - using English queries only")
    
    # Load COSQA from HuggingFace
    corpus, queries, qrels = load_data_from_hf("cosqa")
    print(f"Loaded from HuggingFace: {len(corpus)} corpus, {len(queries)} queries")
    
    # Create Indonesian queries
    queries_indonesian = {}
    for qid, qtext in queries.items():
        if qid in translations:
            queries_indonesian[qid] = translations[qid]
        else:
            queries_indonesian[qid] = qtext
    
    return corpus, queries, queries_indonesian, qrels

# Load data
corpus, queries_en, queries_id, qrels = load_cosqa_data()

## Initialize Models

In [ ]:
# Initialize expander
expander = CrossLingualEmbeddingExpander(
    model_name="intfloat/multilingual-e5-small"
)
print("Expander initialized")

# Initialize retriever
retriever = DenseRetriever(
    model_name="intfloat/multilingual-e5-small",
    device="cpu"
)
print("Retriever initialized")

In [ ]:
def run_embedding_qe(queries, corpus, top_k=10):
    """Run embedding-based query expansion."""
    results = []
    
    corpus_list = list(corpus.values())
    
    for qid, query in tqdm(queries.items(), desc="Processing queries"):
        # Expand query
        expansion = expander.expand(query, num_terms=5)
        
        # Retrieve
        retrieved = retriever.retrieve(
            queries=[expansion.expanded_query],
            corpus=corpus_list,
            top_k=top_k,
        )
        
        results.append({
            "qid": qid,
            "query": query,
            "expanded_query": expansion.expanded_query,
            "expansion_terms": expansion.expansion_terms,
            "retrieved": retrieved[0]
        })
    
    return results

print("run_embedding_qe function defined")

## Run on English Queries

In [ ]:
# Run on English queries
print("Running English query expansion...")
results_en = run_embedding_qe(queries_en, corpus, top_k=10)
print(f"Completed {len(results_en)} English queries")

## Run on Indonesian Queries

In [ ]:
# Run on Indonesian queries
print("Running Indonesian query expansion...")
results_id = run_embedding_qe(queries_id, corpus, top_k=10)
print(f"Completed {len(results_id)} Indonesian queries")

## Evaluation

In [ ]:
def evaluate_results(results, qrels, top_k=10):
    """Evaluate results using NDCG and Recall."""
    from collections import defaultdict
    
    hits = 0
    total_relevant = 0
    ndcg_sum = 0
    evaluated = 0
    
    for result in results:
        qid = result["qid"]
        if qid not in qrels:
            continue
            
        relevant_docs = set(qrels[qid].keys())
        total_relevant += len(relevant_docs)
        
        # Get retrieved docs
        retrieved = result["retrieved"][:top_k]
        retrieved_ids = [doc["id"] for doc in retrieved]
        
        # Calculate hits
        hits += len(set(retrieved_ids) & relevant_docs)
        
        # Calculate NDCG
        dcg = 0
        for i, doc_id in enumerate(retrieved_ids):
            if doc_id in relevant_docs:
                dcg += 1 / np.log2(i + 2)
        
        # Calculate IDCG
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_docs), top_k)))
        
        if idcg > 0:
            ndcg_sum += dcg / idcg
            evaluated += 1
    
    recall = hits / total_relevant if total_relevant > 0 else 0
    ndcg = ndcg_sum / evaluated if evaluated > 0 else 0
    
    return {
        "NDCG@10": ndcg,
        "Recall@10": recall,
        "Hits": hits,
        "Total Relevant": total_relevant,
        "Evaluated": evaluated
    }

# Evaluate English results
metrics_en = evaluate_results(results_en, qrels)
print(f"\nEnglish Results:")
print(f"NDCG@10: {metrics_en['NDCG@10']:.4f}")
print(f"Recall@10: {metrics_en['Recall@10']:.4f}")
print(f"Hits: {metrics_en['Hits']}/{metrics_en['Total Relevant']}")

In [ ]:
# Evaluate Indonesian results
metrics_id = evaluate_results(results_id, qrels)
print(f"\nIndonesian Results:")
print(f"NDCG@10: {metrics_id['NDCG@10']:.4f}")
print(f"Recall@10: {metrics_id['Recall@10']:.4f}")
print(f"Hits: {metrics_id['Hits']}/{metrics_id['Total Relevant']}")

## Results Comparison

In [ ]:
# Compare results
print("\n" + "=" * 60)
print("BENCHMARK RESULTS COMPARISON")
print("=" * 60)

print("\n--- English Queries ---")
print(f"NDCG@10: {metrics_en['NDCG@10']:.4f}")
print(f"Recall@10: {metrics_en['Recall@10']:.4f}")

print("\n--- Indonesian (Translated) Queries ---")
print(f"NDCG@10: {metrics_id['NDCG@10']:.4f}")
print(f"Recall@10: {metrics_id['Recall@10']:.4f}")

# Calculate difference
ndcg_diff = metrics_id['NDCG@10'] - metrics_en['NDCG@10']
recall_diff = metrics_id['Recall@10'] - metrics_en['Recall@10']

print("\n--- Difference (Indonesian - English) ---")
print(f"NDCG@10: {ndcg_diff:+.4f}")
print(f"Recall@10: {recall_diff:+.4f}")
print("=" * 60)

## Save Results

In [ ]:
# Save results to JSON
output_data = {
    "method": "embedding",
    "top_k": 10,
    "metrics": {
        "english": metrics_en,
        "indonesian": metrics_id
    },
    "results": {
        "english": results_en,
        "indonesian": results_id
    }
}

with open("cosqa_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print("Results saved to cosqa_benchmark_results.json")

In [ ]:
# Save detailed results to CSV
def save_detailed_results(results, output_path, queries_english):
    """Save detailed results to CSV."""
    rows = []
    for result in results:
        qid = result["qid"]
        query = result["query"]
        expanded = result.get("expanded_query", query)
        
        # Get top 10 retrieved docs
        retrieved = result["retrieved"][:10]
        for rank, doc in enumerate(retrieved, 1):
            rows.append({
                "qid": qid,
                "query_en": queries_english.get(qid, ""),
                "query_id": query,
                "expanded_query": expanded,
                "rank": rank,
                "doc_id": doc["id"],
                "score": doc["score"]
            })
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"Detailed results saved to {output_path}")

# Save CSV for both languages
save_detailed_results(results_en, "cosqa_embedding_english.csv", queries_en)
save_detailed_results(results_id, "cosqa_embedding_indonesian.csv", queries_en)